# Converting coordinates from a csv file to a json file

In [18]:
import json
import os
from datetime import datetime
import pandas as pd
import numpy as np
import rasterio
from pyproj import Transformer       

## Import the raster file to get the metadata

We need the information from the image as the json file is in reference to the origin and transformation of the tif file.

In [49]:
base_dir = 'images/RawImages/'
tif_name = '2024-05-19_mimal_subset.tif'
tif_path = os.path.join(base_dir, tif_name)

with rasterio.open(tif_path) as raster:
    # Read the raster band
    imported_raster = raster.read(1)
    # Get the metadata of the raster
    imported_raster_meta = raster.meta
    # Get the raster transform parameters
    raster_transform = raster.transform

print("Shape of the raster (rows, columns):")
print(imported_raster.shape)
print("\n")

print("Raster metadata:")
print(imported_raster_meta)
print("\n")

print("Affine transformation parameters:")
print(raster_transform)

Shape of the raster (rows, columns):
(14106, 28149)


Raster metadata:
{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': 0.0, 'width': 28149, 'height': 14106, 'count': 4, 'crs': CRS.from_epsg(32753), 'transform': Affine(3.0, 0.0, 350085.0,
       0.0, -3.0, 8516550.0)}


Affine transformation parameters:
| 3.00, 0.00, 350085.00|
| 0.00,-3.00, 8516550.00|
| 0.00, 0.00, 1.00|


## Convert the locations in the csv file to the same projection as the tif file

In [85]:
# Create the reprojection function
coord_transformer = Transformer.from_crs('epsg:4326', 'epsg:32753', always_xy=True)

# Test on sample set of coordinates
x,y = coord_transformer.transform(133.6153775, -13.7976017)
print(x,y)

350330.60260668746 8474226.385882989


## Checking the raster transformation

In [86]:
pixel_column, pixel_row = ~raster_transform * (x, y)
print(pixel_column, pixel_row)

81.86753556248732 14107.871372337453


In [87]:
df = pd.DataFrame({
        'longitude': [120.5, 120.6, 120.7],
        'latitude': [-30.1, -30.2, -30.3],
        'labels': ['tree', 'building', 'car']
    })

In [91]:
# Specify the path to your CSV file
csv_base_dir = f'data/'
csv_name = '2024 Rapid Waterhole Assessment_aligned.csv'
csv_file_path = os.path.join(csv_base_dir, csv_name)

# Read the CSV file into a DataFrame
waterhole_labelled_df = pd.read_csv(csv_file_path)

# Display the first few rows of the DataFrame
print(waterhole_labelled_df.head())

   OBJECTID   Timestamp   Latitude   Longitude  Class        Observer  \
0         1  2024-05-07 -13.657141  134.354081      2  Andrew Hoskins   
1         2  2024-05-07 -13.655816  134.371830      2  Andrew Hoskins   
2         3  2024-05-07 -13.655044  134.393416      3  Andrew Hoskins   
3         4  2024-05-07 -13.659928  134.417495      2  Andrew Hoskins   
4         5  2024-05-07 -13.657286  134.525346      2  Andrew Hoskins   

   ObserverSi Water_Type_Chopper Water_Type_Satellite transectNu permanence  \
0  Back Right                NaN                River        m08          P   
1  Back Right                NaN      River/Billabong        m08          P   
2  Back Right                NaN            Billabong        m08     I or E   
3  Back Right                NaN               Stream        m08     I or E   
4  Back Right                NaN            Billabong        m08     I or E   

     ID Wet_Dry_Chopper Wet_Dry_Satellite  
0   NaN             NaN               Wet 

In [92]:
x_proj, y_proj = coord_transformer.transform(
    waterhole_labelled_df['Longitude'].values, 
    waterhole_labelled_df['Latitude'].values
)

print(x_proj, y_proj)

[430143.17685067 432062.43965287 434396.87455498 437002.42767336
 448666.15966412 458053.23977042 458237.10489792 458987.86848127
 461874.23246752 449507.09670538 445638.36590115 444025.54981718
 440794.70240962 435105.66625879 428903.23024269 425354.55770888
 424196.77749568 423212.16039536 421165.12538497 417110.28161243
 401196.41106128 391625.28131144 373867.42304861 371360.14238773
 358132.66349237 425752.81788484 426812.24890471 429614.87363284
 458855.26207083 449006.87935519 421700.49235191 421251.19090197
 420799.34395841 389044.36386998 406364.40849872 427547.09390463
 436041.47970019 445150.68701575 459923.74064893 514935.59402967
 382948.88660257 381597.26513111 386069.49127455 388823.87049276
 390486.87336337 379662.95731606 446908.44526433 451882.41080017
 453221.3678156  453138.96858241 455277.34374629 463802.47700669
 444476.01486059 369842.88634585 370015.66197925 371192.01958601
 394621.02910232 397881.89751955 367844.88179619 366973.09283339
 366804.6126904  366446.8

## Convert all to pixel coordinates

In [94]:
# Convert to pixel coordinates
pixel_coords = np.array([~raster_transform * (x, y) for x, y in zip(x_proj, y_proj)])
print(pixel_coords)

[[ 26686.05895022   8817.10079774]
 [ 27325.81321762   8766.5911999 ]
 [ 28103.95818499   8736.12407193]
 [ 28972.47589112   8914.05094087]
 [ 32860.38655471   8808.20206133]
 [ 35989.41325681   8864.42543212]
 [ 36050.70163264   8839.14788764]
 [ 36300.95616042   8959.4865288 ]
 [ 37263.07748917   8776.70499074]
 [ 33140.69890179   5016.483244  ]
 [ 31851.12196705   4438.17430934]
 [ 31313.51660573   4988.84589428]
 [ 30236.56746987   5042.52231285]
 [ 28340.22208626   5001.48777388]
 [ 26272.74341423   5053.92841225]
 [ 25089.85256963   5072.13536447]
 [ 24703.92583189   5071.38972459]
 [ 24375.72013179   5075.28584537]
 [ 23693.37512832   4820.58410868]
 [ 22341.76053748   5164.37665637]
 [ 17037.13702043   6415.48225676]
 [ 13846.76043715   5866.8599326 ]
 [  7927.47434954   4260.85006453]
 [  7091.71412924   3766.73343562]
 [  2682.55449746   6407.96713864]
 [ 25222.60596161   8664.03224081]
 [ 25575.7496349    8601.47342981]
 [ 26509.95787761   8764.08043699]
 [ 36256.75402361   

## Add the new columns to the dataframe

In [95]:
# Add the new columns to the dataframe
waterhole_labelled_df['x_proj'] = x_proj
waterhole_labelled_df['y_proj'] = y_proj
waterhole_labelled_df['pixel_col'] = pixel_coords[:, 0]
waterhole_labelled_df['pixel_row'] = pixel_coords[:, 1]

print(waterhole_labelled_df.head())

   OBJECTID   Timestamp   Latitude   Longitude  Class        Observer  \
0         1  2024-05-07 -13.657141  134.354081      2  Andrew Hoskins   
1         2  2024-05-07 -13.655816  134.371830      2  Andrew Hoskins   
2         3  2024-05-07 -13.655044  134.393416      3  Andrew Hoskins   
3         4  2024-05-07 -13.659928  134.417495      2  Andrew Hoskins   
4         5  2024-05-07 -13.657286  134.525346      2  Andrew Hoskins   

   ObserverSi Water_Type_Chopper Water_Type_Satellite transectNu permanence  \
0  Back Right                NaN                River        m08          P   
1  Back Right                NaN      River/Billabong        m08          P   
2  Back Right                NaN            Billabong        m08     I or E   
3  Back Right                NaN               Stream        m08     I or E   
4  Back Right                NaN            Billabong        m08     I or E   

     ID Wet_Dry_Chopper Wet_Dry_Satellite         x_proj        y_proj  \
0   NaN     

In [96]:
def csv_to_labelme(x, y, 
                   
                   labels=None, 
                   image_path=None, 
                   image_height=None, 
                   image_width=None):
    """
    Convert coordinate columns from a DataFrame to LabelMe JSON format.
    
    Args:
        x (pd.Series): Series containing x/longitude coordinates
        y (pd.Series): Series containing y/latitude coordinates
        labels (pd.Series, optional): Series containing point labels. Defaults to None
        image_path (str, optional): Path to the corresponding image file. Defaults to None
        image_height (int, optional): Height of the image in pixels. Defaults to None
        image_width (int, optional): Width of the image in pixels. Defaults to None
        
    Returns:
        dict: LabelMe formatted JSON
    """
    # Validate inputs
    if len(x) != len(y):
        raise ValueError("x and y coordinates must have the same length")
    if labels is not None and len(labels) != len(x):
        raise ValueError("labels must have the same length as coordinates")
    
    # Initialize LabelMe JSON structure
    labelme_json = {
        "version": "5.0.1",
        "flags": {},
        "shapes": [],
        "imagePath": os.path.basename(image_path) if image_path else "",
        "imageData": None,  # LabelMe stores base64 image data here, but we'll leave it empty
        "imageHeight": image_height,
        "imageWidth": image_width
    }
    
    # Convert each point to LabelMe shape
    point_size = 5  # Size of the point representation in pixels
    
    for i in range(len(x)):
        # Skip if coordinates are NaN
        if pd.isna(x[i]) or pd.isna(y[i]):
            continue
            
        shape = {
            "label": str(labels.iloc[i]) if labels is not None else "point",
            "points": [
                [float(x[i]) - point_size, float(y[i]) - point_size],  # Top-left
                [float(x[i]) + point_size, float(y[i]) + point_size]   # Bottom-right
            ],
            "group_id": None,
            "shape_type": "rectangle",
            "flags": {}
        }
        
        labelme_json['shapes'].append(shape)
    
    # Add creation time
    labelme_json['timeStamp'] = datetime.now().isoformat()
    
    return labelme_json

def save_labelme_json(labelme_json, output_path):
    """Save the LabelMe JSON to file."""
    with open(output_path, 'w') as f:
        json.dump(labelme_json, f, indent=2)
        

## Using the csv input file function

In [ ]:
# Convert and save
labelme_json = csv_to_labelme(
    x=waterhole_labelled_df['pixel_col'],
    y=waterhole_labelled_df['pixel_row'],
    labels=df['Class'],
    image_path=tif_path,
    image_height=1080,
    image_width=1920
)
    
save_labelme_json(labelme_json, "images/labels/labelme_annotation_test.json")